In [1]:
import argparse
from pathlib import Path
import dask.config
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client
from tqdm import tqdm
from pathlib import Path


In [2]:
pfs = list(Path(r'E:\tmp\OKX-TradesBooks-BTC-USDT').glob('*.parquet'))

In [3]:
all(pq.ParquetFile(pf).num_row_groups == 1 for pf in pfs)

True

In [4]:
df0 = pq.ParquetFile(pfs[0]).read().to_pandas()

In [14]:
df0.head(1)

,arg,data,ts,ts_book,asks_0_price,asks_0_amount,asks_0_count,bids_0_price,bids_0_amount,bids_0_count,...,bids_8_price,bids_8_amount,bids_8_count,asks_9_price,asks_9_amount,asks_9_count,bids_9_price,bids_9_amount,bids_9_count,ts_diff
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-02-28 16:01:49.560,"{'channel': 'trades', 'instId': 'BTC-USDT'}","[{'instId': 'BTC-USDT', 'px': '84067.9', 'side...",1740758509560,1.740759e+12,84068.0,0.2538,3.0,84067.9,0.30857,5.0,...,84057.9,0.035977,1.0,84073.3,0.000028,2.0,84056.0,0.02844,3.0,57.0


In [15]:
df0.head(10).to_csv('trades_books_head10.csv', index=False)

In [6]:
df0['ts_diff'] = df0['ts'] - df0['ts_book']

In [ ]:
df0['ts_diff'].max()

245036.0

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 设置图形的风格
plt.style.use('seaborn-v0_8-whitegrid')

# 创建一个图形和坐标轴
plt.figure(figsize=(10, 6))

# 直接在列上调用 .hist()
# bins 参数控制柱子的数量，可以调整它来更好地观察分布
df0['ts_diff'].hist(bins=50, color='skyblue', edgecolor='black')

# 添加标题和标签，使图形更具可读性
plt.title('响应时间频率分布图 (Pandas)', fontsize=16)
plt.xlabel('响应时间 (毫秒)', fontsize=12)
plt.ylabel('频率 (次数)', fontsize=12)

# 显示图形
plt.show()

In [ ]:
import ray

@ray.remote
def backtest(strategy_args):
    
    for r in df.itertuples(index=False):
        strategy.step(r)
        
    return strategy.get_result()

In [72]:
import pandas as pd

class OrderFlowImbalanceStrategy:
    """
    A strategy that triggers on significant order flow imbalance and uses
    fixed percentage take-profit/stop-loss for exits, with transaction fees.
    """
    def __init__(self, volume_threshold: float, take_profit_pct: float, stop_loss_pct: float, fee_rate: float):
        """
        Initializes the strategy with its parameters and state.

        :param volume_threshold: The net volume difference required to trigger a trade.
        :param take_profit_pct: The percentage gain at which to close a position (gross profit).
        :param stop_loss_pct: The percentage loss at which to close a position (gross loss).
        :param fee_rate: The transaction fee rate for a single trade (e.g., 0.0008 for 0.08%).
        """
        # --- Parameters ---
        self.volume_threshold = volume_threshold
        self.take_profit_pct = take_profit_pct
        self.stop_loss_pct = stop_loss_pct
        self.fee_rate = fee_rate  # NEW: Added fee rate parameter
        
        # --- State Variables ---
        self.position = 'flat'  # 'flat', 'long', or 'short'
        self.entry_price = 0.0
        self.entry_time = None
        
        # --- Results Logging ---
        self.trades = []

    def step(self, r: pd.Series):
        """
        Processes a single row of the DataFrame (a single time step).
        'r' is a namedtuple from df.itertuples().
        """
        # 1. Check for exit conditions if we are currently in a position
        if self.position != 'flat':
            mid_price = (r.asks_0_price + r.bids_0_price) / 2
            # Note: TP/SL checks are based on gross market movement, which is a common practice.
            # The fee is accounted for upon closing the position.
            gross_pnl_pct = (mid_price / self.entry_price - 1) if self.position == 'long' else (self.entry_price / mid_price - 1)

            # Check for Take Profit or Stop Loss
            if gross_pnl_pct >= self.take_profit_pct or gross_pnl_pct <= -self.stop_loss_pct:
                exit_reason = 'TP' if gross_pnl_pct > 0 else 'SL'
                self._close_position(r.ts, mid_price, exit_reason)
                return # Exit check is done, move to next step

        # 2. If we are flat, check for new entry signals
        if self.position == 'flat':
            # Assuming r.data is already a list of dicts
            trades_data = r.data

            # Calculate buy and sell volume for this event
            buy_volume = sum(float(trade['sz']) for trade in trades_data if trade['side'] == 'buy')
            sell_volume = sum(float(trade['sz']) for trade in trades_data if trade['side'] == 'sell')
            
            net_buy_volume = buy_volume - sell_volume

            # --- Entry Logic ---
            # Added robustness check (buy_volume > 0) to prevent error on empty sequence
            if net_buy_volume > self.volume_threshold and buy_volume > 0:
                # Strong buy signal
                entry_price = max(float(trade['px']) for trade in trades_data if trade['side'] == 'buy')
                self._open_position('long', r.ts, entry_price)
            
            # Added robustness check (sell_volume > 0)
            elif -net_buy_volume > self.volume_threshold and sell_volume > 0:
                # Strong sell signal
                entry_price = min(float(trade['px']) for trade in trades_data if trade['side'] == 'sell')
                self._open_position('short', r.ts, entry_price)

    def _open_position(self, side: str, time: int, price: float):
        self.position = side
        self.entry_price = price
        self.entry_time = time
        # print(f"{pd.to_datetime(time, unit='ms')} | OPEN {side.upper()} | Price: {price:.2f}")

    def _close_position(self, time: int, price: float, reason: str):
        # --- MODIFIED: PnL calculation now includes fees ---
        if self.position == 'long':
            effective_entry = self.entry_price * (1 + self.fee_rate)
            effective_exit = price * (1 - self.fee_rate)
            pnl = effective_exit - effective_entry
            pnl_pct = (effective_exit / effective_entry - 1) * 100
        else:  # short
            effective_entry = self.entry_price * (1 - self.fee_rate)
            effective_exit = price * (1 + self.fee_rate)
            pnl = effective_entry - effective_exit
            pnl_pct = (effective_entry / effective_exit - 1) * 100
        
        self.trades.append({
            'entry_time': self.entry_time,
            'exit_time': time,
            'entry_price': self.entry_price,
            'exit_price': price,
            'side': self.position,
            'pnl': pnl, # This is now net PnL per unit of asset
            'pnl_pct': pnl_pct, # This is now net PnL percentage
            'reason': reason
        })
        
        # print(f"{pd.to_datetime(time, unit='ms')} | CLOSE {self.position.upper()} | Price: {price:.2f} | Net PnL %: {pnl_pct:.4f} | Reason: {reason}")
        
        # Reset state
        self.position = 'flat'
        self.entry_price = 0.0
        self.entry_time = None

    def get_result(self):
        """
        Returns the backtest results as a DataFrame and a summary.
        """
        if not self.trades:
            return "No trades were executed.", None

        trades_df = pd.DataFrame(self.trades)
        trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'], unit='ms')
        trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'], unit='ms')
        
        # --- Summary Statistics ---
        total_trades = len(trades_df)
        win_rate = (trades_df['pnl'] > 0).sum() / total_trades if total_trades > 0 else 0
        # The PnL columns now reflect net results, so the summary is also net.
        total_pnl_pct = trades_df['pnl_pct'].sum()
        avg_pnl_pct = trades_df['pnl_pct'].mean()
        
        summary = {
            # "Total Trades": total_trades,
            "Win Rate": f"{win_rate:.2%}",
            "Total Net PnL (%)": f"{total_pnl_pct:.4f}%",
            "Average Net PnL per Trade (%)": f"{avg_pnl_pct:.4f}%"
        }
        
        return summary, trades_df


In [65]:
# 策略参数可以根据你的研究进行调整
s = OrderFlowImbalanceStrategy(
    volume_threshold=0.0001,
    take_profit_pct=0.05,     #  止盈
    stop_loss_pct=0.01,      #  止损
    fee_rate=0.0008  # 0.08% 交易费率
)

In [66]:
summary, trades_log = backtest(df0, s)

print("\n--- Backtest Summary ---")
print(summary)
print("\n--- Trade Log ---")
print(trades_log)

Backtesting: 100%|██████████| 997551/997551 [00:05<00:00, 185196.00 events/s]


--- Backtest Summary ---
{'Win Rate': '23.08%', 'Total Net PnL (%)': '3.3281%', 'Average Net PnL per Trade (%)': '0.2560%'}

--- Trade Log ---
                entry_time               exit_time  entry_price  exit_price  \
0  2025-02-28 16:01:49.560 2025-02-28 16:41:29.935      84065.2    84918.65   
1  2025-02-28 16:41:29.941 2025-02-28 16:50:38.128      84920.0    84064.85   
2  2025-02-28 16:50:38.135 2025-02-28 18:37:34.912      84059.0    84913.30   
3  2025-02-28 18:37:34.921 2025-02-28 20:17:32.121      84916.6    84044.85   
4  2025-02-28 20:17:32.234 2025-03-02 15:38:03.145      84044.9    88247.70   
5  2025-03-02 15:38:03.148 2025-03-02 16:18:11.342      88247.7    89139.25   
6  2025-03-02 16:18:11.347 2025-03-02 16:35:30.807      89138.1    90049.90   
7  2025-03-02 16:35:30.808 2025-03-02 17:44:59.149      90053.3    94557.95   
8  2025-03-02 17:44:59.195 2025-03-02 18:10:12.406      94558.0    93550.65   
9  2025-03-02 18:10:12.408 2025-03-02 22:55:02.273      93518.2   

In [ ]:
import itertools
import pandas as pd
from tqdm.notebook import tqdm
import re

# 1. 定义要遍历的参数网格
# 您可以根据需要调整这些值以进行更广泛或更精细的搜索
param_grid = {
    'volume_threshold': [i/10000 for i in range(1,11,1)],
    'take_profit_pct': [i/100 for i in range(5,91,5)],
    'stop_loss_pct': [i/1000 for i in range(10,301,5)]
}

# 固定的手续费率
fee_rate = 0.0008

# 2. 生成所有参数组合
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# 3. 存储所有回测结果的列表
all_results = []

print(f"开始进行参数网格搜索，共 {len(param_combinations)} 种组合...")

# 4. 遍历每一种参数组合并执行回测
# 外层循环使用tqdm来显示网格搜索的总体进度
# 注意：内层的backtest函数也会显示自己的进度条，可能会导致输出有些杂乱
for params in tqdm(param_combinations, desc="Grid Search Progress"):
    # 使用当前组合的参数初始化策略
    strategy = OrderFlowImbalanceStrategy(
        volume_threshold=params['volume_threshold'],
        take_profit_pct=params['take_profit_pct'],
        stop_loss_pct=params['stop_loss_pct'],
        fee_rate=fee_rate
    )
    
    # 运行回测
    summary, _ = backtest(df0, strategy)
    
    # 收集结果
    result_row = params.copy()
    if isinstance(summary, dict):  # 检查是否有交易发生
        result_row.update(summary)
    else:  # 如果没有交易，则填充默认值
        result_row.update({
            "Win Rate": "N/A",
            "Total Net PnL (%)": "0.0%",
            "Average Net PnL per Trade (%)": "0.0%"
        })
    all_results.append(result_row)

# 5. 将结果转换为DataFrame并进行处理和排序
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # 将字符串格式的PnL转换为浮点数以便排序
    # 使用正则表达式提取数字，以处理'N/A'等情况
    results_df['pnl_numeric'] = results_df['Total Net PnL (%)'].apply(
        lambda x: float(re.findall(r"[-+]?\d*\.\d+|\d+", str(x))[0]) if re.findall(r"[-+]?\d*\.\d+|\d+", str(x)) else 0.0
    )
    
    # 按总净收益率降序排序
    sorted_results_df = results_df.sort_values(by='pnl_numeric', ascending=False).drop(columns=['pnl_numeric'])
    
    # 6. 打印最终排序后的结果
    print("\n--- 参数网格搜索结果 (按总净PnL降序排列) ---")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(sorted_results_df)
else:
    print("网格搜索完成，但没有记录到任何回测结果。")

开始进行参数网格搜索，共 10620 种组合...


Grid Search Progress:   0%|          | 0/10620 [00:00<?, ?it/s]

In [ ]:
# 导入所需的库
import ray
import itertools
import pandas as pd
from tqdm.notebook import tqdm
import re

# --- Ray 并行化改造 ---

# 1. 初始化 Ray。ignore_reinit_error=True 在 notebook 中很有用，可以避免重复执行时出错。
# 您可以根据需要配置 num_cpus，默认会使用所有可用的 CPU 核心。
ray.init(ignore_reinit_error=True, num_cpus=8)

# 2. 定义一个远程函数，该函数封装了单次回测的全部逻辑。
# @ray.remote 装饰器使其可以被 Ray 并行调度。
@ray.remote
def run_backtest_task(params, df_ref, fee_rate):
    """
    一个 Ray 任务，用于执行单个参数组合的回测。
    
    :param params: 包含策略参数的字典。
    :param df_ref: 共享在 Ray 对象存储中的数据帧的引用 (ObjectRef)。
    :param fee_rate: 交易费率。
    :return: 包含参数和回测结果的字典。
    """
    # OrderFlowImbalanceStrategy 和 backtest 函数是在之前的单元格中定义的，
    # Ray 会将它们的定义传递给工作进程。
    strategy = OrderFlowImbalanceStrategy(
        volume_threshold=params['volume_threshold'],
        take_profit_pct=params['take_profit_pct'],
        stop_loss_pct=params['stop_loss_pct'],
        fee_rate=fee_rate
    )
    
    # 直接在任务中从引用获取数据
    df = ray.get(df_ref)
    
    # 运行回测
    summary, _ = backtest(df, strategy)
    
    # 收集结果，与原代码逻辑相同
    result_row = params.copy()
    if isinstance(summary, dict):  # 检查是否有交易发生
        result_row.update(summary)
    else:  # 如果没有交易，则填充默认值
        result_row.update({
            "Win Rate": "N/A",
            "Total Net PnL (%)": "0.0%",
            "Average Net PnL per Trade (%)": "0.0%"
        })
    return result_row


# --- 参数设置 (与原代码相同) ---

# 定义要遍历的参数网格
param_grid = {
    'volume_threshold': [i/10000 for i in range(1, 11, 1)],
    'take_profit_pct': [i/100 for i in range(5, 91, 5)],
    'stop_loss_pct': [i/1000 for i in range(10, 301, 5)]
}

# 固定的手续费率
fee_rate = 0.0008

# 生成所有参数组合
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]


# --- 执行并行回测 ---

# 3. 将大型数据 df0 放入 Ray 的共享对象存储，返回一个引用。
# 这可以极大地提高效率，避免在每个任务中重复序列化和传输数据。
df0_ref = ray.put(df0)

print(f"开始使用 Ray 进行参数网格搜索，共 {len(param_combinations)} 种组合...")

# 4. 并行启动所有回测任务。
# .remote() 会立即返回一个 ObjectRef (对未来结果的引用)，而不会阻塞等待。
result_futures = [run_backtest_task.remote(params, df0_ref, fee_rate) for params in param_combinations]

# 5. 使用 tqdm 跟踪任务完成进度，并用 ray.get() 获取结果。
# 此循环会按顺序等待每个任务完成并获取其结果。
all_results = [ray.get(future) for future in tqdm(result_futures, desc="Grid Search Progress")]


# --- 资源清理与结果处理 ---

# 6. 完成计算后关闭 Ray
ray.shutdown()

# 7. 将结果转换为DataFrame并进行处理和排序 (与原代码相同)
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # 将字符串格式的PnL转换为浮点数以便排序
    results_df['pnl_numeric'] = results_df['Total Net PnL (%)'].apply(
        lambda x: float(re.findall(r"[-+]?\\d*\\.\\d+|\\d+", str(x))[0]) if re.findall(r"[-+]?\\d*\\.\\d+|\\d+", str(x)) else 0.0
    )
    
    # 按总净收益率降序排序
    sorted_results_df = results_df.sort_values(by='pnl_numeric', ascending=False).drop(columns=['pnl_numeric'])
    
    # 8. 打印最终排序后的结果
    print("\n--- 参数网格搜索结果 (按总净PnL降序排列) ---")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(sorted_results_df)
else:
    print("网格搜索完成，但没有记录到任何回测结果。")